# Lab 7: Compare three policy recipes

**Goal:** Compare ACT, learner SmolVLA, and Diffusion Policy on the same bowl-plate task. Read every frozen final-state attempt. Keep the source demonstrations, simulator states, reset seeds, success predicate, and timeout fixed. State the differences in pretraining, processor, training work, and inference.

**Inputs:** The committed comparison config, 90 saved final attempts, training resource record, and the published SmolVLA prompt study. **Compute:** CPU for this report. **Cloud cost:** US$0 on the prepared host. Start Jupyter from the repository root. Read the [lab guide](README.md).

## 1. Check the frozen comparison

The recipe was committed before the final states were opened. Its SHA-256 must match the report. The three methods used final states 20–49 in the same order. An infrastructure or policy error would invalidate this comparison. The source demonstrations may overlap simulator initial states; this course has no proof of a new-placement test.

In [ ]:
from pathlib import Path
import hashlib
import json
import yaml
import matplotlib.pyplot as plt
import numpy as np

WORKSPACE = Path.cwd().resolve()
CONFIG = WORKSPACE / 'configs/experiments/comparison.yaml'
REPORT = WORKSPACE / 'artifacts/reference/comparison/final/three_method_report.json'
RESOURCES = WORKSPACE / 'artifacts/reference/comparison/training_resources.json'
assert CONFIG.is_file() and REPORT.is_file(), 'Start Jupyter from the repository root and install the reference results.'
config = yaml.safe_load(CONFIG.read_text())
report = json.loads(REPORT.read_text())
resources = json.loads(RESOURCES.read_text())
config_hash = hashlib.sha256(CONFIG.read_bytes()).hexdigest()
assert report['comparison_config_sha256'] == config_hash
assert report['state_ids'] == config['state_ids'] == list(range(20, 50))
assert report['comparison_type'] == 'practical_recipe'
assert set(report['methods']) == {'act', 'smolvla_learner', 'diffusion'}
print('Frozen config SHA-256:', config_hash)
print('Final state IDs:', report['state_ids'][0], 'to', report['state_ids'][-1])

## 2. Compare task success with uncertainty

Read the count and its 95% Wilson interval. The interval describes uncertainty for this fixed set under a binomial model. State IDs can share scene structure and may overlap source demonstrations, so the interval is not a claim about all robot scenes. Keep all 30 attempts in each denominator.

In [ ]:
names = ['act', 'smolvla_learner', 'diffusion']
labels = ['ACT', 'SmolVLA C2', 'Diffusion']
for name, label in zip(names, labels, strict=True):
    item = report['methods'][name]
    lower, upper = item['wilson_95']
    print(f"{label:13s} {item['successes']:2d}/{item['attempts']}  95% Wilson [{lower:.3f}, {upper:.3f}]")

rates = [report['methods'][name]['success_rate'] for name in names]
low = [rates[i] - report['methods'][name]['wilson_95'][0] for i, name in enumerate(names)]
high = [max(0.0, report['methods'][name]['wilson_95'][1] - rates[i]) for i, name in enumerate(names)]
fig, axis = plt.subplots(figsize=(7, 4))
axis.bar(labels, rates, color=['#4c78a8', '#f58518', '#54a24b'])
axis.errorbar(range(3), rates, yerr=[low, high], fmt='none', ecolor='black', capsize=5)
axis.set(ylabel='Success fraction', ylim=(0, 1.08), title='Frozen bowl-plate final states (30 each)')
plt.show()

## 3. Inspect paired states and failures

The methods saw the same state IDs. A paired difference counts states where only one method succeeded. The exact McNemar value is descriptive for these selected policies and states. It does not isolate model architecture. Read task failures and controller limit events from the raw rows.

In [ ]:
rows = {}
for name in names:
    result_dir = WORKSPACE / report['methods'][name]['result_dir']
    summary = json.loads((result_dir / 'summary.json').read_text())
    rows[name] = [json.loads(line) for line in (result_dir / 'episodes.jsonl').read_text().splitlines()]
    assert summary['valid_for_comparison'] and len(rows[name]) == 30
    assert [row['initial_state_id'] for row in rows[name]] == report['state_ids']
    assert all(
        row['partition'] == 'final'
        and row['task_id'] == config['task_alias']
        and row['metrics']['seed'] == config['base_seed'] + row['initial_state_id']
        and row['policy_revision'] == report['methods'][name]['model_sha256']
        and row['error'] is None
        for row in rows[name]
    )

matrix = np.array([[int(row['success']) for row in rows[name]] for name in names])
fig, axis = plt.subplots(figsize=(12, 2.6))
axis.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
axis.set(yticks=range(3), yticklabels=labels, xticks=range(30), xticklabels=report['state_ids'], xlabel='Frozen final state ID')
axis.tick_params(axis='x', labelrotation=90)
plt.show()
for key, pair in report['pairs'].items():
    print(key, 'only first:', pair['a_only'], 'only second:', pair['b_only'],
          'difference:', round(pair['paired_difference'], 3),
          'exact McNemar p:', round(pair['exact_mcnemar_p'], 4))

## 4. Compare data use, training work, and runtime

All three recipes use the same 39 new robot demonstrations. The number of optimizer steps, sampled windows, and pretrained data differ. A repeated window counts each time it enters a batch. The recorded training time excludes downloads, setup, and evaluation. The episode time includes simulator startup and steps; it is not action-generation latency. The recorded GPU memory probes used different conditions, so this lab does not rank the models by peak VRAM.

In [ ]:
assert resources['training_episode_count'] == config['train_episode_count'] == 39
for name, label in zip(names, labels, strict=True):
    train = resources['methods'][name]
    result = report['methods'][name]
    print(label)
    print('  optimizer steps:', train['robot_training_steps'],
          'sampled windows:', train['sampled_windows'],
          'loop seconds:', round(train['optimizer_loop_s'], 3))
    print('  source:', train['pretraining'])
    print('  median final episode seconds:', round(result['median_episode_elapsed_s'], 3),
          'clipped action-value fraction:', round(result['clipped_action_value_fraction'], 4))
assert report['methods']['act']['robot_training_steps'] < report['methods']['diffusion']['robot_training_steps']
assert report['methods']['diffusion']['robot_training_steps'] < report['methods']['smolvla_learner']['robot_training_steps']

## 5. Separate prompt sensitivity from task selection

The published SmolVLA B in Lab 4 is a separate model. Its training data are unknown. Its prompt study used one development state and kept the bowl-plate success predicate fixed. A changed prompt changed the outcome, but this one state does not prove general language understanding. ACT and Diffusion Policy need an external choice of task-specific checkpoint for a new goal. That choice is an oracle, not language understanding by either policy.

In [ ]:
published = yaml.safe_load((WORKSPACE / 'manifests/smolvla_reference.yaml').read_text())['published_reference']
study = published['prompt_study']
for label, key in [('canonical', 'canonical_result'), ('paraphrase', 'paraphrase_result'), ('conflicting', 'conflicting_result')]:
    result = json.loads((WORKSPACE / study[key] / 'summary.json').read_text())
    print(label, 'prompt:', result['model_prompt'], 'goal:', result['evaluation_task'],
          'success:', result['successes'], '/', result['episodes'])
    assert result['state_ids'] == [0] and result['partition'] == 'development'
print('Third-task gate:', (WORKSPACE / 'artifacts/reference/comparison/third_task_gate.json').exists())

## 6. Make a task-specific choice

For this fixed bowl-plate setup, the prepared ACT recipe succeeded on every final state and used the least recorded robot training time. Select it for this task if task success and local training cost are the main constraints. This choice does not prove that ACT is better for a different task, visual shift, language goal, or physical robot. Do not use the final states to tune a checkpoint after this report.

**Reading block (20 minutes):** Read [RT-2 Section 3.2, Robot-Action Fine-tuning](https://arxiv.org/html/2307.15818#S3.SS2). RT-2 turns discretized robot actions into output tokens and limits robot-action decoding to valid tokens. Predict what could fail if this limit were removed. Then state why the SmolVLA flow action expert in this course is not an RT-2 action-token decoder.

In [ ]:
act = report['methods']['act']
assert act['successes'] == act['attempts'] == 30
print('Selected practical recipe for this fixed task: ACT')
print('Model SHA-256:', act['model_sha256'])
print('Open limits: source demonstration overlap, training-seed variance, visual shift, language task selection, and physical transfer.')

## Completion check

Save the [exercise](exercise.md) before you read the [solution guide](../../solutions/07-comparison.md). Your answer must name a method for the stated task, show its exact evidence, and list at least one condition where this result does not apply.